In [29]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data          # valeur numérique
        self.grad = 0.0           # gradient
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op            # opération (pour debug)
        
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [30]:
a = Value(2.0)
b = Value(3.0)
c = a*b + b

c.backward()

print(a.grad)
print(b.grad)

3.0
3.0


In [31]:
def relu(self):
    out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

    def _backward():
        self.grad += (out.data > 0) * out.grad

    out._backward = _backward
    return out